<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/Filtrado_ATS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Instalar librerías requeridas si no están presentes
!pip install -q pypdf sentence-transformers

In [ ]:
# ROBOT DE ANÁLISIS DE CURRÍCULUMS CON INTELIGENCIA ARTIFICIAL
# Google Colab | NLP + Embeddings semánticos

In [ ]:
import os
import re
import numpy as np
import pandas as pd

from pypdf import PdfReader
from google.colab import files
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# ------------------------------------------------------------
# 1. CONFIGURAR LA VACANTE
# ------------------------------------------------------------

puesto = input("Nombre del puesto: ")

print("\nDescribe el perfil requerido:")
perfil = input("Perfil: ")

print("\nEscribe los requisitos INDISPENSABLES, uno por línea.")
print("Ejemplo: Experiencia en Python")

requisitos_obligatorios = [
    x.strip()
    for x in input(
        "Requisitos indispensables separados por punto y coma: "
    ).split(";")
    if x.strip()
]

print("\nEscribe los requisitos DESEABLES separados por punto y coma.")
requisitos_deseables = [
    x.strip()
    for x in input("Requisitos deseables: ").split(";")
    if x.strip()
]

# Evitar una evaluación sin criterios concretos
if not requisitos_obligatorios and not requisitos_deseables:
    raise ValueError(
        "Define al menos un requisito para evaluar la vacante."
    )


Nombre del puesto: Analista de datos

Describe el perfil requerido:
Perfil: Profesional responsable de analizar información, desarrollar reportes, validar datos y automatizar procesos utilizando herramientas de análisis.

Escribe los requisitos INDISPENSABLES, uno por línea.
Ejemplo: Experiencia en Python
Requisitos indispensables separados por punto y coma: Experiencia en SQL; Manejo de Python; Análisis de datos

Escribe los requisitos DESEABLES separados por punto y coma.
Requisitos deseables: Power BI; Machine Learning; Automatización de reportes


In [ ]:
# ------------------------------------------------------------
# 2. CARGAR MODELO MULTILINGÜE
# ------------------------------------------------------------

print("\nCargando modelo de lenguaje...")

modelo = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

print("Modelo cargado correctamente.")


Cargando modelo de lenguaje...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo cargado correctamente.


In [ ]:
# ------------------------------------------------------------
# 3. SUBIR CURRÍCULUMS PDF
# ------------------------------------------------------------

print("\nSelecciona los currículums en PDF:")
archivos = files.upload()

# ------------------------------------------------------------
# 4. EXTRAER TEXTO DE LOS PDF
# ------------------------------------------------------------

def leer_pdf(ruta):
    """Extrae el texto de un currículum PDF."""
    try:
        lector = PdfReader(ruta)

        if lector.is_encrypted:
            return ""

        paginas = []
        for pagina in lector.pages:
            paginas.append(pagina.extract_text() or "")

        return "\n".join(paginas).strip()

    except Exception as error:
        print(f"Error al leer {ruta}: {error}")
        return ""


# Dividir documentos extensos en fragmentos.
# Así se reduce el riesgo de perder información relevante.
def dividir_texto(texto, max_caracteres=1800):
    palabras = texto.split()
    fragmentos = []
    actual = []
    longitud = 0

    for palabra in palabras:
        if longitud + len(palabra) + 1 > max_caracteres:
            if actual:
                fragmentos.append(" ".join(actual))
            actual = [palabra]
            longitud = len(palabra)
        else:
            actual.append(palabra)
            longitud += len(palabra) + 1

    if actual:
        fragmentos.append(" ".join(actual))

    return fragmentos


# ------------------------------------------------------------
# 5. PREPARAR CRITERIOS DE EVALUACIÓN
# ------------------------------------------------------------

# Los requisitos indispensables tienen mayor peso.
criterios = (
    [(r, 2.0, "Indispensable")
     for r in requisitos_obligatorios]
    +
    [(r, 1.0, "Deseable")
     for r in requisitos_deseables]
)

# Incluir el perfil general como contexto adicional.
contexto_vacante = perfil.strip()

# Codificar criterios una sola vez.
textos_criterios = [
    contexto_vacante + ". Requisito: " + criterio
    for criterio, peso, tipo in criterios
]

embeddings_criterios = modelo.encode(
    textos_criterios,
    normalize_embeddings=True,
    show_progress_bar=False
)

# ------------------------------------------------------------
# 6. EVALUAR CADA CURRÍCULUM
# ------------------------------------------------------------

resultados = []
detalle_requisitos = []

# Umbral inicial configurable; requiere calibración con ejemplos.
UMBRAL_REQUISITO = 0.40

for nombre_archivo in archivos.keys():

    print(f"\nAnalizando: {nombre_archivo}")

    texto = leer_pdf(nombre_archivo)

    if len(texto.strip()) < 50:
        resultados.append({
            "Archivo": nombre_archivo,
            "Puntuacion": np.nan,
            "Requisitos_indispensables": "No evaluables",
            "Requisitos_faltantes": "PDF sin texto suficiente",
            "Recomendacion": "Revisión manual del documento"
        })
        continue

    fragmentos = dividir_texto(texto)

    # Embeddings del contenido del currículum.
    embeddings_cv = modelo.encode(
        fragmentos,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    # Similitud semántica entre cada criterio y cada fragmento.
    matriz_similitud = cosine_similarity(
        embeddings_criterios,
        embeddings_cv
    )

    # Para cada criterio, conservar la mejor coincidencia.
    mejores_similitudes = np.max(matriz_similitud, axis=1)

    # Escalar a 0-100 para facilitar la lectura.
    # Es una puntuación heurística, NO una probabilidad.
    puntuaciones = np.clip(
        mejores_similitudes * 100, 0, 100
    )

    pesos = np.array([peso for _, peso, _ in criterios])

    puntuacion_global = np.average(
        puntuaciones, weights=pesos
    )

    faltantes = []
    detalle = []

    for i, (criterio, peso, tipo) in enumerate(criterios):

        similitud = float(mejores_similitudes[i])

        detalle.append({
            "Archivo": nombre_archivo,
            "Tipo": tipo,
            "Requisito": criterio,
            "Coincidencia_semantica": round(
                float(puntuaciones[i]), 2
            ),
            "Requiere_verificacion": (
                "Sí" if similitud < UMBRAL_REQUISITO
                else "Verificar evidencia"
            )
        })

        if (
            tipo == "Indispensable"
            and similitud < UMBRAL_REQUISITO
        ):
            faltantes.append(criterio)

    if faltantes:
        recomendacion = "Revisar requisitos indispensables"
    elif puntuacion_global >= 65:
        recomendacion = "Priorizar revisión humana"
    else:
        recomendacion = "Revisión humana complementaria"

    resultados.append({
        "Archivo": nombre_archivo,
        "Puntuacion": round(float(puntuacion_global), 2),
        "Requisitos_indispensables": (
            "Por verificar" if requisitos_obligatorios
            else "Sin requisitos definidos"
        ),
        "Requisitos_faltantes": (
            "; ".join(faltantes) if faltantes
            else "Sin alertas por umbral semántico"
        ),
        "Recomendacion": recomendacion
    })

    detalle_requisitos.extend(detalle)

# ------------------------------------------------------------
# 7. MOSTRAR RESULTADOS
# ------------------------------------------------------------

df_resultados = pd.DataFrame(resultados)

if not df_resultados.empty:
    df_resultados = df_resultados.sort_values(
        "Puntuacion",
        ascending=False,
        na_position="last"
    )

print("\n" + "=" * 65)
print("RESULTADOS DEL ROBOT DE RECLUTAMIENTO")
print("Vacante:", puesto)
print("=" * 65)

display(df_resultados)

df_detalle = pd.DataFrame(detalle_requisitos)

if not df_detalle.empty:
    print("\nDETALLE DE COINCIDENCIAS POR REQUISITO")
    display(df_detalle)

# ------------------------------------------------------------
# 8. EXPORTAR REPORTES
# ------------------------------------------------------------

df_resultados.to_csv(
    "reporte_candidatos.csv",
    index=False,
    encoding="utf-8-sig"
)

df_detalle.to_csv(
    "detalle_requisitos.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nReportes generados.")

files.download("reporte_candidatos.csv")
files.download("detalle_requisitos.csv")


Selecciona los currículums en PDF:


Saving CURRICULUM VITAE (1).pdf to CURRICULUM VITAE (1).pdf

Analizando: CURRICULUM VITAE (1).pdf

RESULTADOS DEL ROBOT DE RECLUTAMIENTO
Vacante: Analista de datos


,Archivo,Puntuacion,Requisitos_indispensables,Requisitos_faltantes,Recomendacion
0,CURRICULUM VITAE (1).pdf,51.57,Por verificar,Sin alertas por umbral semántico,Revisión humana complementaria



DETALLE DE COINCIDENCIAS POR REQUISITO


,Archivo,Tipo,Requisito,Coincidencia_semantica,Requiere_verificacion
0,CURRICULUM VITAE (1).pdf,Indispensable,Experiencia en SQL,50.12,Verificar evidencia
1,CURRICULUM VITAE (1).pdf,Indispensable,Manejo de Python,53.19,Verificar evidencia
2,CURRICULUM VITAE (1).pdf,Indispensable,Análisis de datos,52.25,Verificar evidencia
3,CURRICULUM VITAE (1).pdf,Deseable,Power BI,52.34,Verificar evidencia
4,CURRICULUM VITAE (1).pdf,Deseable,Machine Learning,49.70,Verificar evidencia
5,CURRICULUM VITAE (1).pdf,Deseable,Automatización de reportes,50.95,Verificar evidencia



Reportes generados.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>